In [ ]:
# 1. Clone code từ GitHub (Thay bằng link repo của bạn nếu khác)
!git clone https://github.com/ThanhTrunggDEV/GenAI.git
%cd GenAI
!ls -F

# Hướng dẫn chi tiết
1. **GitHub**: Đảm bảo code mới nhất đã được push lên GitHub.
2. **Kaggle**: Tạo notebook mới, bật **Internet** và chọn Accelerator là **GPU T4 x2** (hoặc P100).
3. **Chạy**: Chạy lần lượt các cell bên dưới.

# Hmong Pattern AI - Kaggle Training Notebook

Notebook này hướng dẫn cách train mô hình AI tạo hoa văn Hmong trên Kaggle sử dụng GPU miễn phí (T4 x2 hoặc P100).

## 1. Kiểm tra phần cứng
Đảm bảo Accelerator được bật là **GPU T4 x2** hoặc **P100**.

In [ ]:
# Cài đặt dependencies (thêm --ignore-installed để tránh Conflict)
!pip install -q diffusers transformers accelerate bitsandbytes
# Sửa lỗi conflict version bằng cách cài đặt cụ thể
!pip install -q packaging>=23.2.0,<26.0.0 fastcore==1.8.0
!pip install -q -r requirements.txt

## 2. Cài đặt môi trường
Cài đặt các thư viện cần thiết.

In [ ]:
# Set PYTHONPATH để python tìm thấy module motif
import os
os.environ['PYTHONPATH'] = os.getcwd()

# Chuẩn bị dữ liệu training
!python -m motif.data.prepare

# Trích xuất đặc trưng (Stage 1)
!python -m motif.models.visual_encoder
!python -m motif.models.cultural_encoder
!python -m motif.models.combine_embeddings

## 3. Chuẩn bị dữ liệu
Giả sử bạn đã upload dataset hoặc mount code từ github. 
Chạy các bước chuẩn bị dữ liệu và encode đặc trưng văn hóa.

In [ ]:
!accelerate launch --num_processes=1 train_diffusion.py \
    --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
    --output_dir="outputs/hmong-pattern-lora" \
    --num_train_epochs=10 \
    --train_batch_size=4

## 4. Train Model (Stage 2)
Chạy script training trên GPU. Thời gian train khoảng 2-4 tiếng cho 5000 steps.

In [ ]:
!accelerate launch train_diffusion.py \
    --pretrained_model_name_or_path="stabilityai/stable-diffusion-2-1-base" \
    --output_dir="outputs/hmong-pattern-lora" \
    --num_train_epochs=10 \
    --train_batch_size=4

## 5. Tạo mẫu thử nghiệm (Inference)
Sử dụng model vừa train để tạo ảnh mới.

In [ ]:
!python -m motif.pipeline.generate --prompt "Hmong spiral pattern in indigo" --checkpoint "outputs/hmong-pattern-lora"

## 6. Lưu kết quả
Nén folder output để tải về.

In [ ]:
!zip -r outputs.zip outputs/